<a href="https://colab.research.google.com/github/n3cx011/Nature-Inspired-Feature-Selection-for-Breast-Cancer-Diagnosis/blob/main/ITBNM_2313_0024_PSO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from google.colab import files


uploaded = files.upload()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

X_train = pd.read_csv('X_train_scaled.csv')
X_test = pd.read_csv('X_test_scaled.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

In [ ]:
!pip install scikit-opt

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sko.PSO import PSO

In [ ]:
def objective_function(x):

    binary_x = (np.array(x) > 0.5).astype(int)
    selected_features = np.where(binary_x == 1)[0]


    if len(selected_features) == 0:
        return 1.0

    xtrain_sub = X_train.iloc[:, selected_features]
    xtest_sub = X_test.iloc[:, selected_features]

    clf = KNeighborsClassifier(n_neighbors=5)
    clf.fit(xtrain_sub, y_train)
    preds = clf.predict(xtest_sub)

    error = 1.0 - accuracy_score(y_test, preds)
    feature_ratio = len(selected_features) / X_train.shape[1]


    alpha = 0.99
    beta = 0.01

    cost = (alpha * error) + (beta * feature_ratio)
    return cost

In [ ]:
n_features = X_train.shape[1]



pso = PSO(func=objective_function,
          n_dim=n_features,
          pop=40,
          max_iter=30,
          lb=[0]*n_features,
          ub=[1]*n_features,
          w=0.8, c1=0.5, c2=0.5)


best_x, best_y = pso.run()


best_binary_x = (np.array(best_x) > 0.5).astype(int)

print("PSO Optimization complete!")
print(f"Best feature vector (1=keep, 0=discard): {best_binary_x}")

In [ ]:
pso_selected_indices = np.where(best_binary_x == 1)[0]
pso_selected_feature_names = X_train.columns[pso_selected_indices]

print(f"Total features selected by PSO: {len(pso_selected_indices)} out of {n_features}")
print(f"Selected feature names: list({pso_selected_feature_names})")


X_train_pso = X_train.iloc[:, pso_selected_indices]
X_test_pso = X_test.iloc[:, pso_selected_indices]

final_pso_clf = KNeighborsClassifier(n_neighbors=5)
final_pso_clf.fit(X_train_pso, y_train)
pso_preds = final_pso_clf.predict(X_test_pso)


from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("\n--- PSO MODEL PERFORMANCE ---")
print(f"Accuracy:  {accuracy_score(y_test, pso_preds):.4f}")
print(f"Precision: {precision_score(y_test, pso_preds):.4f}")
print(f"Recall:    {recall_score(y_test, pso_preds):.4f}")
print(f"F1-Score:  {f1_score(y_test, pso_preds):.4f}")

In [ ]:
pso_results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Feature Count'],
    'PSO_Score': [
        accuracy_score(y_test, pso_preds),
        precision_score(y_test, pso_preds),
        recall_score(y_test, pso_preds),
        f1_score(y_test, pso_preds),
        len(pso_selected_indices)
    ]
})
pso_results.to_csv('pso_performance_results.csv', index=False)
files.download('pso_performance_results.csv')